
Prueba rápida en modo “01 ya está descargando”:

1. Abre Terminal 1 (Agente 02):

```sh
# en C./
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\tsis_agents\run_agent02_quotes_strict_loop.ps1" -RunId "20260305_quotes_session_01" -QuotesRoot "C:\TSIS_Data\data\quotes_p95" -MaxFiles 500 -SleepSec 15
# en D:/
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\tsis_agents\run_agent02_quotes_strict_loop.ps1" -RunId "20260305_quotes_session_D01" -QuotesRoot "D:\quotes" -MaxFiles 500 -SleepSec 15

[agent02] RunId=20260305_quotes_session_01
[agent02] QuotesRoot=C:\TSIS_Data\data\quotes_p95
[agent02] RunDir=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260305_quotes_session_01
[2026-03-05 18:04:41] {"processed_total": 5550, "pending": 591774, "pass": 1917, "soft": 2998, "hard": 635, "retry": 3633, "gate": "NO_CLOSE_RETRY_PENDING", "run_dir": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\20260305_quotes_session_01"}
[2026-03-05 18:06:25] {"processed_total": 6050, "pending": 591274, "pass": 1926, "soft": 3489, "hard": 635, "retry": 4124, "gate": "NO_CLOSE_RETRY_PENDING", "run_dir": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\20260305_quotes_session_01"}
```
2. Abre Terminal 2 (Agente 03 rápido, instantáneo):

```sh
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\tsis_agents\run_agent03_live_fast.ps1" -RunId "20260305_quotes_session_01" -IntervalSec 5


agent03-live-fast
time: 2026-03-05 18:09:47
run : 20260305_quotes_session_01
json: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260305_quotes_session_01\live_status_quotes_strict.json
----------------------------------------------------------------------------------------------------
updated_utc                : 2026-03-05T17:09:28.749925+00:00
probe_root                 : C:\TSIS_Data\data\quotes_p95
max_files                  : 500
files_discovered_total     : 596824
files_pending              : 590774
files_processed_total_state: 6550
files_current_snapshot     : 6550
retry_pending_files_current: 4524

severity_counts_current:
  - HARD_FAIL: 669
  - PASS: 2026
  - SOFT_FAIL: 3855

top_causes_current:
  - crossed_rows_present_but_under_threshold: 3844
  - crossed_ratio_gt_threshold: 667
  - crossed_ratio_gt_hard_cap: 35
  - dtype_mismatch: 19
  - ask_integer_with_crossed_anomaly: 7
  - missing_required_columns: 2
  - zero_rows: 2

```
3. Abre Terminal 3 (Agente 03 analítico, cada 2 min):
```sh
# version conpacta
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\tsis_agents\run_agent03_monitor_compact.ps1" -RunId "20260305_quotes_session_01" -SleepSec 60

[agent03-compact] RunId=20260305_quotes_session_01
[agent03-compact] RunDir=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260305_quotes_session_01
[2026-03-05 18:05:35] {"updated_utc": "2026-03-05T17:02:59.276265+00:00", "processed": 5050, "retry_pending": 3133, "hard_fail": 624, "gate": "NO_CLOSE_RETRY_PENDING", "mean_coverage_ok": 0.0}
[2026-03-05 18:06:36] {"updated_utc": "2026-03-05T17:06:10.795645+00:00", "processed": 5550, "retry_pending": 3633, "hard_fail": 635, "gate": "NO_CLOSE_RETRY_PENDING", "mean_coverage_ok": 0.0}
```

Checks en 2 minutos:

- Debe existir:
    - ...\20260305_quotes_session_01\quotes_agent_strict_events_current.csv
    - ...\20260305_quotes_session_01\live_status_quotes_strict.json
- En terminal fast (03_live_fast) debe subir:
    - files_processed_total_state
    - files_current_snapshot
    - severity_counts_current

Si no sube nada, revisa que -QuotesRoot sea exactamente la carpeta donde 01 escribe quotes.parquet.



## Whatcher

“watcher” adicional que:

- lea esos artefactos cada X segundos,
- detecte anomalías (stalled, hard fail spike, retry creciendo),
- te saque alertas claras en consola.


- C:\Users\AlexJ\Desktop\tsis_agents\run_agent_supervisor.ps1

Comando (una línea):

```sh
powershell -ExecutionPolicy Bypass -File "C:\Users\AlexJ\Desktop\tsis_agents\run_agent_supervisor.ps1" -RunId "20260305_quotes_session_01" -IntervalSec 10 -StallSec 180 -HardFailSpike 50 -RetrySpike 200
```
Qué vigila:

- live_status_quotes_strict.json freshness (STALL)
- progreso (NO_PROGRESS)
- picos HARD_FAIL y retry_pending
- top causas actuales
- existencia de events_current y retry_current